# HyDE RAG v2: domain-adapt the embedder (F2) on a free Colab GPU

1. Generates 3 synthetic queries per food-corpus chunk with `doc2query/msmarco-t5-base-v1`
   (Apache-2.0). This is a different model and prompt from the eval set; no eval questions are used.
2. Mines one hard negative per query: ranks 10-50 under bge-base, skipping the positive's own page.
3. Fine-tunes `BAAI/bge-base-en-v1.5` with cached MultipleNegativesRankingLoss (1 epoch, lr 2e-5,
   batch 64 computed in chunks of 16, which fits a free 15 GB T4).
4. Encodes the corpus with the base and the fine-tuned model.

**How to run**
1. On the laptop: `uv run hyde-rag export-food-colab` creates `data\food\colab_upload\food_chunks.zip`.
2. Here: *Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*. Upload the zip when asked.
   This takes about 20-30 minutes.
3. Three zips download at the end. Extract them on the laptop:
   - `food-indexes.zip` into `data\indexes\`
   - `bge-base-food-model.zip` into `models\`
   - `food-synthetic.zip` into `data\food\synthetic\`
4. Check: `uv run hyde-rag verify-index --dataset food --encoder bge-base` (and `--encoder bge-base-food`).

**If a cell fails** (e.g. CUDA out of memory): *Runtime -> Restart session*, then *Run all* again.
Finished steps (the uploaded chunks, the base encoding, the synthetic queries) are reused from disk.

The training code below is `src/hyde_rag/train/finetune_core.py` from the repo, unchanged.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Same versions as the project (pyproject.toml).
!pip -q install sentence-transformers==6.1.0 transformers==5.19.0 accelerate==1.15.0 datasets==5.1.0

In [ ]:
import os
import zipfile

from google.colab import files

if not os.path.exists("food/chunks.jsonl"):
    uploaded = files.upload()  # choose food_chunks.zip
    with zipfile.ZipFile(next(iter(uploaded))) as zf:
        zf.extractall("food")
with open("food/chunks.jsonl", encoding="utf-8") as f:
    print(sum(1 for _ in f), "chunks")

In [ ]:
# --- src/hyde_rag/train/finetune_core.py (verbatim) ---
"""Domain adaptation F2: fine-tune bge on synthetic queries with mined hard negatives.

Self-contained (torch / transformers / sentence-transformers only): this file is embedded
verbatim into notebooks/finetune_food_embedder_colab.ipynb so the free-GPU run uses exactly the
code tested here. Never trains on eval questions: queries are generated from corpus chunks
by a different model (doc2query T5) with a different prompt than the eval set.
"""

from __future__ import annotations

import json
import random

import numpy as np
import torch

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
DOC2QUERY = "doc2query/msmarco-t5-base-v1"  # Apache-2.0


def index_text(title: str, section: str, text: str) -> str:
    """Same as DocChunk.index_text in the project: title and section path as a header."""
    header = f"{title} | {section}" if section else title
    return f"{header}\n{text}"


def load_food_chunks(chunks_path: str, docs_path: str) -> tuple[list[str], list[str], list[str], list[str]]:
    """-> chunk_ids, index texts, raw texts, group keys (doc_id:page) for negative filtering."""
    titles = {}
    with open(docs_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                d = json.loads(line)
                titles[d["doc_id"]] = d["title"]
    ids, idx_texts, raw, groups = [], [], [], []
    with open(chunks_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            c = json.loads(line)
            ids.append(c["chunk_id"])
            idx_texts.append(index_text(titles[c["doc_id"]], c["section"], c["text"]))
            raw.append(c["text"])
            groups.append(f"{c['doc_id']}:{c['page']}")
    return ids, idx_texts, raw, groups


def generate_queries(
    texts: list[str],
    n_per_text: int = 3,
    model_id: str = DOC2QUERY,
    batch_size: int = 32,
    device: str = "cuda",
    seed: int = 13,
    max_input_tokens: int = 384,
) -> list[list[str]]:
    """Sampled doc2query queries per text (top-p 0.95, top-k 10), deduplicated."""
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device).eval()
    out: list[list[str]] = []
    with torch.inference_mode():
        for s in range(0, len(texts), batch_size):
            batch = tok(
                texts[s : s + batch_size],
                truncation=True,
                max_length=max_input_tokens,
                padding=True,
                return_tensors="pt",
            ).to(device)
            gen = model.generate(
                **batch, max_length=64, do_sample=True, top_p=0.95, top_k=10, num_return_sequences=n_per_text
            )
            dec = tok.batch_decode(gen, skip_special_tokens=True)
            for i in range(len(dec) // n_per_text):
                qs = [q.strip() for q in dec[i * n_per_text : (i + 1) * n_per_text] if q.strip()]
                out.append(list(dict.fromkeys(qs)))
    return out


def encode(model, texts: list[str], prefix: str = "", batch_size: int = 64) -> np.ndarray:
    """Normalised embeddings from a SentenceTransformer (bge: CLS pooling + Normalize)."""
    return model.encode(
        [prefix + t for t in texts],
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def mine_negatives(
    query_vecs: np.ndarray,
    corpus_vecs: np.ndarray,
    positives: list[int],
    groups: list[str],
    rank_min: int = 10,
    rank_max: int = 50,
    seed: int = 13,
) -> list[int]:
    """One hard negative per query: a random corpus item ranked rank_min..rank_max for that query,
    excluding items from the positive's own page (likely false negatives)."""
    rng = random.Random(seed)
    q = torch.from_numpy(query_vecs)
    c = torch.from_numpy(corpus_vecs)
    negatives = []
    for start in range(0, len(q), 1024):
        top = torch.topk(q[start : start + 1024] @ c.T, k=min(rank_max, len(c)), dim=1).indices.numpy()
        for row, cand in enumerate(top):
            pos = positives[start + row]
            pool = [int(j) for j in cand[rank_min:rank_max] if groups[int(j)] != groups[pos]]
            negatives.append(rng.choice(pool) if pool else -1)
    return negatives


def train(
    base_model: str,
    anchors: list[str],
    positives: list[str],
    negatives: list[str],
    out_dir: str,
    epochs: int = 1,
    batch_size: int = 64,
    lr: float = 2e-5,
    query_prefix: str = BGE_QUERY_PREFIX,
    fp16: bool = True,
    seed: int = 13,
    mini_batch_size: int = 16,
) -> str:
    """Cached MultipleNegativesRankingLoss (GradCache) on (anchor, positive, hard negative) triplets.

    Same loss and in-batch negatives as MNRL at batch_size, but embeddings are computed
    mini_batch_size examples at a time, so 64 triplets of 512-token chunks fit on a free 15 GB T4
    (plain MNRL ran out of memory there).
    """
    from datasets import Dataset
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
    )
    from sentence_transformers.base.sampler import BatchSamplers
    from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss

    model = SentenceTransformer(base_model)
    data = Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})
    args = SentenceTransformerTrainingArguments(
        output_dir=out_dir + "-checkpoints",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_steps=0.1,  # a float is a ratio in transformers v5
        fp16=fp16,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        prompts={"anchor": query_prefix},  # bge queries carry the instruction, passages don't
        seed=seed,
        save_strategy="no",
        logging_steps=50,
        report_to="none",
    )
    loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=mini_batch_size)
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=loss)
    trainer.train()
    model.save(out_dir)
    return out_dir

In [ ]:
import gc
import time  # json, numpy, torch: imported by the module cell above
from pathlib import Path

from sentence_transformers import SentenceTransformer

torch.backends.cuda.matmul.allow_tf32 = False
ids, idx_texts, raw, groups = load_food_chunks("food/chunks.jsonl", "food/docs.jsonl")
BASE = "BAAI/bge-base-en-v1.5"


def save_index(name, vecs, model_id):
    d = Path("out/food") / name
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "embeddings.npy", vecs.astype(np.float32))
    (d / "doc_ids.json").write_text(json.dumps(ids))
    meta = {
        "name": name,
        "model_id": model_id,
        "pooling": "cls",
        "normalize": True,
        "query_prefix": BGE_QUERY_PREFIX,
        "max_length": 512,
        "device": torch.cuda.get_device_name(0),
    }
    (d / "meta.json").write_text(json.dumps(meta, indent=2))


t0 = time.time()
base = SentenceTransformer(BASE, device="cuda")
base.max_seq_length = 512
if Path("out/food/bge-base/embeddings.npy").exists():  # re-run after a restart: reuse it
    corpus_base = np.load("out/food/bge-base/embeddings.npy")
else:
    corpus_base = encode(base, idx_texts)
    save_index("bge-base", corpus_base, BASE)
print(f"base corpus encoded: {corpus_base.shape} in {time.time() - t0:.0f}s")

In [ ]:
t0 = time.time()
qfile = Path("out/synthetic/queries.jsonl")
if qfile.exists():  # re-run after a restart: reuse the generated queries
    row_of = {cid: i for i, cid in enumerate(ids)}
    with open(qfile, encoding="utf-8") as f:
        pairs = [(r["query"], row_of[r["chunk_id"]]) for r in map(json.loads, f)]
else:
    queries = generate_queries(raw, n_per_text=3)
    pairs = [(q, i) for i, qs in enumerate(queries) for q in qs]
    qfile.parent.mkdir(parents=True, exist_ok=True)
    with open(qfile, "w", encoding="utf-8") as f:
        for q, i in pairs:
            f.write(json.dumps({"chunk_id": ids[i], "query": q}) + "\n")
print(f"{len(pairs)} synthetic queries in {time.time() - t0:.0f}s; e.g. {pairs[0][0]!r}")

In [ ]:
qv = encode(base, [q for q, _ in pairs], prefix=BGE_QUERY_PREFIX)
neg = mine_negatives(qv, corpus_base, [i for _, i in pairs], groups)
keep = [k for k, n in enumerate(neg) if n >= 0]
anchors = [pairs[k][0] for k in keep]
positives = [idx_texts[pairs[k][1]] for k in keep]
negatives = [idx_texts[neg[k]] for k in keep]
print(f"{len(keep)} training triplets")
del base, qv
gc.collect()
torch.cuda.empty_cache()  # free the GPU for training

In [ ]:
t0 = time.time()
model_dir = train(BASE, anchors, positives, negatives, "out/models/bge-base-food")
print(f"fine-tuned in {(time.time() - t0) / 60:.1f} min -> {model_dir}")
tuned = SentenceTransformer(model_dir, device="cuda")
tuned.max_seq_length = 512
save_index("bge-base-food", encode(tuned, idx_texts), "models/bge-base-food")

In [ ]:
import shutil

for name, root, base_dir in [
    ("food-indexes", "out", "food"),
    ("bge-base-food-model", "out/models", "bge-base-food"),
    ("food-synthetic", "out/synthetic", "."),
]:
    path = shutil.make_archive(name, "zip", root_dir=root, base_dir=base_dir)
    print(path, round(os.path.getsize(path) / 1e6), "MB")
    files.download(path)